# Assignment 10: MT19937 Stream Cipher and Predictable Token Attack

## Objective
Understand why MT19937 is unsuitable for cryptography through a toy stream cipher, 16-bit seed recovery, and timestamp-token prediction.

## 1. MT19937 explanation
MT19937 maintains a 624-word state and applies shifts, tempering, and permutation to generate statistically uniform values. It is deterministic and invertible, so output observations can reveal its state. It is not cryptographically secure: it has no secret-key property, is predictable, and lacks backtracking resistance.

## 2. Known plaintext and 16-bit seed search
For a candidate seed, the known `A` bytes reveal keystream positions: `keystream = ciphertext XOR known_plaintext`. For this assignment, the brute-force method tries every seed from 0 through 65,535. This is 2^16 = 65,536 candidates.

### Part A recovery pseudocode
```text
for seed in 0 through 65535:
    plaintext = MT19937_XOR(ciphertext, seed)
    if plaintext is printable, prefix length is 5..20,
       and plaintext ends with 14 'A' characters:
        return seed, plaintext
```

In [ ]:
import sys
sys.path.insert(0, '..')
from analysis.mt19937_assignment import mt19937_xor, run_assignment
message = b'Cryptography Lab Assignment 10'
seed = 0x1234
ciphertext = mt19937_xor(message, seed)
decrypted = mt19937_xor(ciphertext, seed)
print('Plaintext:', message.decode())
print('Ciphertext (hex):', ciphertext.hex())
print('Decrypted:', decrypted.decode())
print('Round trip verified:', decrypted == message)


In [ ]:
result = run_assignment(prefix_length=12)
recovery = result['part_a_seed_recovery']
print('Selected seed:', recovery['selected_seed'])
print('Plaintext:', recovery['plaintext'])
print('Ciphertext (hex):', recovery['ciphertext_hex'])
print('Recovered seed:', recovery['recovered_seed'])
print('Full plaintext verified:', recovery['full_plaintext_verified'])
print('Known 14-A sequence verified:', recovery['known_sequence_verified'])


## 3. Predictable timestamp reset tokens
The application seeds MT19937 with Unix timestamp `T` and generates the first 16 8-bit outputs. An attacker searching `[T-300, T+300]` tests 601 timestamps. A timestamp has low entropy and may be guessed from request/log timing, so it is not suitable for token generation. Use `secrets.token_bytes(16)` or another cryptographically secure CSPRNG.

### Token prediction pseudocode
```text
observed = attacker-observed 16-byte token
for candidate in T-300 through T+300:
    generated = first_16_bytes_after_seeding_MT19937(candidate)
    if generated == observed:
        report candidate
```

In [ ]:
prediction = result['part_b_token_prediction']
print('Generation timestamp T:', prediction['timestamp'])
print('Search interval:', prediction['interval'])
print('Candidates tested:', prediction['candidates_tested'])
print('Observed token (hex):', prediction['observed_token_hex'])
print('Matching timestamp(s):', prediction['matches'])
print('Timestamp verified:', prediction['timestamp_verified'])
print('Regenerated token (hex):', prediction['regenerated_token_hex'])


## 4. Security conclusion
MT19937 is suitable for games, simulations, and testing, but reset tokens should use a CSPRNG. All values in this demonstration are self-generated for the lab; no real application data was used.